# DP codebase: validation and published reproductions

Use this notebook to verify implementations against Jaeger, Couzin, Lymburn, and Topaz results. It includes parameter sweeps and saved full-scale artifacts and is not intended as a first tutorial.

The notebook suite is split by purpose:

- `Tutorial_ABM.ipynb` — student ABM quickstart.
- `Tutorial_ESN.ipynb` — student ESN quickstart.
- `Tutorial_swarmRC.ipynb` — student swarm-reservoir quickstart.
- `advanced/Existing_Model_Validation.ipynb` — validation of published models and full reproductions.

All use the same codebase. Start with the tutorial matching your task; move to the specialist notebooks when your project needs additional evidence or diagnostics.


Saved figures/animations from this notebook go into topic-named subfolders (FIGURES/ESN, FIGURES/ABM, FIGURES/swarmRC, and the matching ANIMATIONS/ subfolders, created below), so re-running cells does not scatter loose files across DP/.

In [ ]:
# This notebook lives in advanced/, while source paths are relative to the repository root.
isdir("TIME_SERIES") || cd("..")
isdir("TIME_SERIES") || error("Open this notebook from inside the DP_student repository.")

for d in ("FIGURES/ESN", "FIGURES/ABM", "FIGURES/swarmRC", "ANIMATIONS/ABM", "ANIMATIONS/swarmRC")
    mkpath(d)
end

## 0. Model systems: what kind of data are you looking at?

Before training a reservoir, separate three objects that are often casually
called an "attractor":

1. A **time series** is one or more measured coordinates against time.
2. A **state-space attractor** is the trajectory in the system's actual state
   variables. It is available in simulations because we know every variable.
3. A **delay embedding** reconstructs geometry from one scalar observation,
   for example `(x(t), x(t-τ), x(t-2τ))`. It is an observation-derived
   representation, not an additional dynamical system and not automatically
   a valid embedding for arbitrary choices of dimension and lag.

A projection can hide dimensions. This matters especially for
Mackey–Glass, whose state at time `t` is an entire history segment over
`[t-τ,t]`, and for hyperchaos, where a two- or three-dimensional projection
cannot show all expanding directions.

### Systems available in `TIME_SERIES/my_systems.jl`

| System | Type / state | Use it when… | Main caution |
|---|---|---|---|
| Logistic map | discrete map, 1D | you want a cheap sanity check of data flow, return maps or one-step prediction | samples are iterations, not a continuous flow |
| Rössler | continuous ODE, 3D | you want clean single-scroll geometry, a localized fold, or a first embedding exercise | it is visually easier than Lorenz and not representative of every chaotic flow |
| Lorenz | continuous ODE, 3D | you want harder switching dynamics, strong local divergence, or a standard chaotic benchmark | sampling rate strongly changes apparent predictability and measured memory |
| Hyperchaotic Rössler | continuous ODE, 4D | one unstable direction is not enough and you need a genuinely higher-dimensional prediction, reconstruction, synchronization or control problem | the [standard equations and parameters](https://www.scholarpedia.org/article/Hyperchaos) produce two positive Lyapunov exponents; a 2D/3D projection cannot display both directly |
| Mackey–Glass | delay differential equation; effectively infinite-dimensional state | delayed feedback or learning from a scalar record is the scientific issue | a few delay coordinates are only a finite-dimensional view of the history state |

In [ ]:
include("TIME_SERIES/my_systems.jl")
include("TIME_SERIES/my_time_series_analysis.jl")

log_demo = logistic_data(r=3.9, x0=0.2, T=2500)
lor_demo = lorenz_data(tspan=(0.0, 80.0), dt_data=0.05, u0=(1.0, 1.0, 1.0))
ros_demo = rossler_data(tspan=(0.0, 220.0), dt_data=0.05)
hros_demo = hyper_rossler_data(tspan=(0.0, 400.0), dt_data=0.05)
mg_demo = mackey_glass_data(tau=17, n_out=3000, discard=1500)

# Uniformly sampled views for plotting (lorenz_data retains its adaptive ODE solution).
t_lor = collect(20.0:lor_demo.dt_data:80.0)
X_lor = Array(lor_demo.sol(t_lor))
keep_ros = ros_demo.t .>= 20.0
keep_hros = hros_demo.t .>= 100.0

model_data = (
    logistic=log_demo.data,
    lorenz=X_lor,
    rossler=ros_demo.data[:, keep_ros],
    hyper_rossler=hros_demo.data[:, keep_hros],
    mackey_glass=mg_demo.data,
)

[(name, size(data)) for (name, data) in pairs(model_data)]

In [ ]:
# Each row shows: measured signal, natural/full-state view, scalar reconstruction.
fig_systems = Figure(size=(1450, 1550))

function timeseries_panel!(slot, x, title; nshow=600)
    n = min(nshow, length(x))
    ax = Axis(slot, xlabel="sample", ylabel="observed x", title=title)
    lines!(ax, 1:n, x[1:n]; linewidth=1)
end

function embedding_panel!(slot, x, lag, title; connected=true)
    ax = Axis(slot, xlabel="x(n)", ylabel="x(n+$lag)", title=title, aspect=DataAspect())
    if connected
        lines!(ax, x[1:end-lag], x[1+lag:end]; linewidth=0.7)
    else
        scatter!(ax, x[1:end-lag], x[1+lag:end]; markersize=2)
    end
end

# Logistic map
xlog = vec(model_data.logistic)[501:end]
timeseries_panel!(fig_systems[1, 1], xlog, "Logistic: scalar time series")
ax = Axis(fig_systems[1, 2], xlabel="x(n)", ylabel="x(n+1)", title="Known map relation", aspect=DataAspect())
scatter!(ax, xlog[1:end-1], xlog[2:end]; markersize=2)
embedding_panel!(fig_systems[1, 3], xlog, 1, "Lag-1 reconstruction"; connected=false)

# Lorenz flow
X = model_data.lorenz; x = vec(X[1, :])
timeseries_panel!(fig_systems[2, 1], x, "Lorenz: x(t)")
ax3 = Axis3(fig_systems[2, 2], xlabel="x", ylabel="y", zlabel="z", title="Full 3D state-space attractor")
lines!(ax3, X[1, :], X[2, :], X[3, :]; linewidth=0.5)
# dt=0.05 and lag=3 gives Δt=0.15: a compact illustrative view that
# avoids the visibly over-unfolded geometry produced here by lag=10.
embedding_panel!(fig_systems[2, 3], x, 3, "Scalar lag reconstruction (Δt=0.15)")

# Rössler flow
X = model_data.rossler; x = vec(X[1, :])
timeseries_panel!(fig_systems[3, 1], x, "Rössler: x(t)")
ax3 = Axis3(fig_systems[3, 2], xlabel="x", ylabel="y", zlabel="z", title="Full 3D state-space attractor")
lines!(ax3, X[1, :], X[2, :], X[3, :]; linewidth=0.5)
embedding_panel!(fig_systems[3, 3], x, 30, "Scalar lag reconstruction")

# Four-dimensional hyperchaotic Rössler: only projections fit on a page.
X = model_data.hyper_rossler; x = vec(X[1, :])
timeseries_panel!(fig_systems[4, 1], x, "Hyper-Rössler: x(t)")
ax3 = Axis3(fig_systems[4, 2], xlabel="x", ylabel="y", zlabel="z", title="3D projection of 4D state")
lines!(ax3, X[1, :], X[2, :], X[3, :]; linewidth=0.4)
embedding_panel!(fig_systems[4, 3], x, 20, "Scalar lag reconstruction")

# Mackey–Glass: the true state is a history function, so both are projections.
xmg = vec(model_data.mackey_glass)
timeseries_panel!(fig_systems[5, 1], xmg, "Mackey–Glass: scalar time series")
ax = Axis(fig_systems[5, 2], xlabel="y(n)", ylabel="y(n-17)",
    title="Two-coordinate history-state projection", aspect=DataAspect())
lines!(ax, xmg[18:end], xmg[1:end-17]; linewidth=0.7)
embedding_panel!(fig_systems[5, 3], xmg, 15, "Jaeger's lag-15 attractor view")

fig_systems

The right-hand column deliberately uses convenient illustrative lags, not
automatically optimal embeddings. For inference from real scalar data,
choose the lag (for example from autocorrelation or mutual information),
then choose embedding dimension (for example with false nearest
neighbours), and test robustness. `takens_embed` constructs the coordinates;
it does not certify that the chosen coordinates form an embedding.

## A. Echo state network: build it, verify it, then use it

Both A1 and A2 use a fixed random recurrent network and train only a linear
readout, so both are ESNs. They are not the same ESN experiment:

| | A1: Jaeger reproduction | A2: generic codebase ESN |
|---|---|---|
| Task | autonomously regenerate Mackey–Glass | learn one-step Lorenz dynamics, then optionally free-run |
| Drive during training | previous true output through `Wback` plus constant bias | current external input through `Win` |
| Drive during generation | its own predicted output through `Wback` | its own predicted state supplied as the next external input |
| State update | independent `C` and decay `a` | one conventional `leak` parameter |
| Reservoir matrix | dense Jaeger-specific construction | sparse configurable construction |
| Readout target | one scalar | all three Lorenz coordinates |
| Main validation | published `NRMSE84` and autonomous attractor | prediction plus memory, stability, separability and feature health |
| Implementation | `JaegerFeedbackESN` | `BasicESN <: AbstractReservoir` |

**A1** reproduces Jaeger (2001)'s own headline Mackey-Glass benchmark, the
result that established echo state networks as a serious method rather
than just an architecture description. **A2** then builds a plain ESN
(`RC/my_esn.jl`) driven by a Lorenz signal and walks through its
diagnostics (memory capacity, stability, separability, feature health),
useful for understanding *any* reservoir built in this codebase, not just
the specific network from A1.

### Which Jaeger results can we use as checks?

There are three distinct levels of evidence here:

- **Task-level quantitative reproduction (A1):** Section 6 of Jaeger
  (2001) reports `NRMSE84≈0.00028` for τ=17 with 3000 training points.
  This is stronger evidence than the visually similar attractor and is
  already computed and asserted below.
- **Echo-state/stability property (A2):** the 2001 report establishes
  contraction/forgetting as the core idea and gives spectral conditions.
  The perturbation experiment below checks the operational consequence:
  two nearby reservoir states receiving the same input should converge.
  This is a property check, not a reproduction of one published number;
  spectral radius below one is a useful construction heuristic, not by
  itself a universal certificate for a nonlinear driven ESN.
- **Linear short-term memory (A2):** the familiar `MC=ΣR²_k≤N` result is
  from Jaeger's separate 2002 report,
  [*Short Term Memory in Echo State Networks*](https://publica.fraunhofer.de/entities/publication/9dfaead1-4dc0-4e3c-b89b-596f50f671c1),
  not the 2001 Mackey–Glass section. The explicit i.i.d.-input
  check below reproduces that bound on a small ESN.

Thus the confidence ladder is: published prediction error, autonomous
attractor geometry, perturbation forgetting, and an independent memory
capacity bound. They test different failure modes and should not be
collapsed into one claim that an ESN "works".

### A1. Reproducing Jaeger (2001): the Mackey-Glass benchmark

Section 6 of Jaeger's original report ("The 'echo state' approach to
analysing and training recurrent neural networks", GMD Report 148,
[PDF](https://www.ai.rug.nl/minds/uploads/EchoStatesTechRep.pdf)) trains
an ESN to autonomously regenerate the Mackey-Glass chaotic attractor, and
reports its 84-step-ahead prediction error (`NRMSE84`) against the
published literature: roughly two orders of magnitude better than prior
methods at the time. This is the field's own reference result for what an
ESN can do, not an arbitrary demonstration. Jaeger's Fig. 14 (τ=17)
shows, from left to right, the original system and networks trained on
21000 and 3000 steps: "the three plots are visually indistinguishable".
The report's printed caption incorrectly calls the third panel "bottom";
it is plainly the right-hand panel, so that caption is omitted here.

See Fig. 14 in [Jaeger's original report](https://doi.org/10.24406/publica-fhg-291111). The source figure is not redistributed with this repository.

**The task, precisely** (Jaeger eqns 22-23): the Mackey-Glass delay
differential equation `ẏ(t) = 0.2·y(t-τ)/(1+y(t-τ)¹⁰) - 0.1·y(t)`, discretized
with step `δ=0.1` then subsampled by 10 so one output step is one unit time
interval, `τ=17` (the "mildly chaotic" case; "the majority of studies" use
this value. `τ=30` is also in the paper but needs careful noise-tuning for
stability that this reproduction doesn't attempt). Sequences are squashed
into `[-1,1]` via `y ↦ tanh(y-1)` before use.

**The architecture is not a plain ESN**: a 400-unit reservoir, a *leaky*
update with independent time-constant `C=0.44` and decay rate `a=0.9`
(`RC/my_jaeger_esn.jl`'s `JaegerFeedbackESN`; this needs its own
implementation, since `RC/my_esn.jl`'s `BasicESN` only has a single leak
parameter), a constant bias input `u(n)=0.2`, and, the key structural
difference, **output feedback**: the network's own (teacher-forced, then
generated) output feeds back into the reservoir through `Wback`, rather
than an external input driving it. This is why the task is "regenerate the
attractor autonomously", not "predict from an external signal".
The learned object is therefore an autonomous scalar time-series model.
Jaeger's "attractor plot" is a two-coordinate delay view of that generated
series, plotting `(y(n), y(n+15))`; the embedding is the visualisation, not
a separate target learned by the ESN.

**Evaluation follows Jaeger's Section 6.3 protocol exactly**: 50
independent 1000-step teacher-forced + 84-step free-run trials, comparing
the free-run prediction at step 84 against ground truth in the *original*
(un-squashed) coordinates, normalised by Jaeger's own stated variance of
the attractor signal (`σ²≈0.067`) so the result is directly comparable to
his number.

In [ ]:
include("TIME_SERIES/my_systems.jl")
include("RC/my_jaeger_esn.jl")

jaeger_mg = jaeger_mackey_glass_reproduction(tau=17, train_len=3000, seed=1)

println("NRMSE84 (this reproduction): ", round(jaeger_mg.nrmse84, digits=5))
println("NRMSE84 (Jaeger 2001, τ=17, 3000-step training): ≈0.00028")
println("Best non-ESN methods Jaeger compares against: NRMSE84 ≈ 0.0088-0.032")

# Not expected to match Jaeger's exact hand-tuned figure (~1 hour of manual
# parameter search on his own random matrices, which this reproduction
# doesn't repeat). The acceptance bar is "clearly, dramatically better
# than the non-ESN baselines above", which is the actual headline claim.
@assert jaeger_mg.nrmse84 < 0.01 "NRMSE84 should be well below the non-ESN baselines (~0.0088-0.032), got $(jaeger_mg.nrmse84)"
println("\naccepted: NRMSE84 < 0.01 (non-ESN baseline range)")

Two sanity checks worth looking at directly rather than trusting one
scalar: do individual 84-step predictions actually track the true
attractor point-by-point, and does the trained network stay on the
attractor over a much longer free run (not just 84 steps)?

In [ ]:
fig_mg = Figure(size=(1450, 350))

ax1 = Axis(fig_mg[1, 1], xlabel="trial", ylabel="y(n+84)",
    title="NRMSE84 trials: truth vs. prediction")
scatter!(ax1, 1:jaeger_mg.eval.n_trials, jaeger_mg.eval.truths; label="truth", markersize=8)
scatter!(ax1, 1:jaeger_mg.eval.n_trials, jaeger_mg.eval.preds; label="predicted", markersize=6)
axislegend(ax1; position=:rb)

# Longer free run: teacher-force 1000 steps, then generate 1500 steps freely
# (Jaeger's own Fig. 15 check for τ=17: "predictions start to deviate
# perceptibly ... not earlier than about 1200 steps").
mg_long = mackey_glass_data(tau=17, n_out=1000 + 1500, discard=2000)
y_long = vec(mg_long.data)
Y_long = reshape(mg_squash(y_long), 1, :)

function mg_free_run_demo(res, Wout, Y_long, y_long)
    reset!(res)
    y_prev = zeros(1)
    for n in 1:1000
        step!(res, y_prev; noise_scale=0.0)
        y_prev = Y_long[:, n]
    end
    gen = zeros(1500)
    for n in 1:1500
        step!(res, y_prev; noise_scale=0.0)
        y_prev = vec(Wout * res.x)
        gen[n] = mg_unsquash(y_prev)[1]
    end
    return gen
end
gen = mg_free_run_demo(jaeger_mg.res, jaeger_mg.Wout, Y_long, y_long)

ax2 = Axis(fig_mg[1, 2], xlabel="step (free-run)", ylabel="y",
    title="1500-step free run vs. truth (τ=17)")
lines!(ax2, 1:1500, y_long[1001:end]; label="truth", linewidth=1.5)
lines!(ax2, 1:1500, gen; label="generated", linewidth=1.0, linestyle=:dash)
axislegend(ax2; position=:rb)

# Jaeger's Fig. 14 visualisation: a lag-15 coordinate view of the scalar output.
embed_lag = 15
truth_free = y_long[1001:end]
ax3 = Axis(fig_mg[1, 3], xlabel="y(n)", ylabel="y(n+15)",
    title="Learned attractor (lag-15 view)", aspect=DataAspect())
lines!(ax3, truth_free[1:end-embed_lag], truth_free[1+embed_lag:end];
    label="original", linewidth=1.5)
lines!(ax3, gen[1:end-embed_lag], gen[1+embed_lag:end];
    label="generated", linewidth=1.0, linestyle=:dash)
axislegend(ax3; position=:rb)

fig_mg

### A2. Build your own ESN and read its diagnostics

`A1`'s network is a specific, purpose-built architecture for one
benchmark: output feedback, two-parameter leaky integration, no external
input. Most uses of an ESN in this codebase (and in Part C) look
different: a plain input-driven reservoir (`RC/my_esn.jl`'s `BasicESN`),
trained on whatever signal you want it to learn, read through the shared
diagnostic suite in `RC/my_reservoir_core.jl`. That's what the rest of
this section walks through: how to build one, and how to tell whether
it's any good.

By default we drive it with a Lorenz signal (real structure to learn); a
`driver` toggle further down also lets you swap in pure i.i.d. noise
instead, a useful contrast when interpreting the diagnostics later in
this section (several of the notes below explicitly suggest trying it).

## B. Couzin swarms in 2D and 3D

Two geometries, run through the same six-step pipeline: (1) build and
simulate one instance, (2) pick a validated instance of each of Couzin's
four named regimes, (3) sweep `(Δr_o, Δr_a)` and look at the resulting
`(p_group, m_group)` phase structure, (4) regime snapshots and
animations, (5) collective-order diagnostics, (6) interaction-network
topology. B1 is a periodic 2D adaptation, fast to iterate on; B2 is the
paper's own unbounded 3D geometry.

**Only B2 is checked directly against Couzin (2002) Fig. 3**, since that
figure is itself a 3D result: B1's own sweep (Step 3) runs the same full
grid-and-ensemble machinery on the 2D adaptation and is worth seeing in
its own right, but a 2D adaptation isn't the thing Fig. 3 was measured
from, so it isn't scored against it. Persistent-homology (TDA) analysis
of Couzin swarms lives in its own section, Section D, applied once
there rather than repeated per geometry here.

Either B1 or B2 can be run independently: B2 does not depend on B1
having been run first.

> **First pass:** run one 2D regime in B1.1 and one 3D regime in B2.1.
> Phase sweeps, interaction networks and fluidity diagnostics are
> independent project tools, not prerequisites for Part C.

### B1. Two-dimensional periodic Couzin model

#### B1.1 Construct and simulate

A compact 2D model: positions/velocities are 2D, and distances use the
minimum-image convention across a periodic square. Repulsion/orientation/
attraction zones, the blind rear cone, finite turning speed and constant
forward speed follow Couzin et al. (2002); this periodic-box version is a
fast adaptation for iteration, not the paper's own (unbounded) geometry --
see Step 2 onward for the paper-faithful version.

In [ ]:
include("ABM/load_ABM.jl")
include("ABM/models/Couzin/load_Couzin.jl")

scenario_b = Couzin_params_from_preset(:milling; N=50, L=50.0, dt=0.1)
P_b = scenario_b.P
simcfg_b = SimulationConfig(steps=1000, dt=scenario_b.dt, seed=1)

out_b = simulate_Couzin_2d(simcfg_b, P_b)
(length(out_b.pos_hist), length(out_b.t))   # (frames stored, timepoints)

#### B1.2 Validated instances of Couzin's four named regimes

Couzin (2002) labels four collective behaviours by where `(Δr_o, Δr_a)`
falls: **swarm** (low `p_group`, low `m_group`), **torus/milling** (low
`p_group`, high `m_group`), **dynamic parallel group** (high `p_group`, low
`m_group`, but fluid: density and neighbours keep changing), **highly
parallel group** (very high `p_group`, very low `m_group`, rigid).
`COUZIN_REGIME_PRESETS` (`ABM/models/Couzin/my_Couzin_core.jl`) gives a
`(Zr, Zo, Za)` point for each regime; `preset_accepts` is the numeric
acceptance test (thresholds on `p_group`/`m_group`/dilation) that Couzin's
own description implies.

A preset landing in the right region on average doesn't guarantee every
single seed does. `generate_validated_preset` reruns a preset across seeds
until one actually passes `preset_accepts`, and returns that run directly.
Its default `collect_history=true` means these runs can be reused directly
for the snapshots/animations in Step 4, with no need to re-simulate.

**Periodic, not unbounded:** the paper's own geometry is unbounded, but
with purely local (finite-range) interactions and no boundary, a group
that thins out even slightly can lose contact entirely once neighbours
exceed `Za` apart. There is nothing to pull them back. Checked directly
(group radius over time, not just the order parameters): several
intermediate-polarisation points that looked plausible on paper grew
20-30x in radius over 5000 steps instead of settling (genuine dispersal
toward fragmentation, not the density fluctuation Couzin (2002)
describes). Periodic boundaries (`simulate_Couzin_2d`, `periodic=true`)
keep the group bounded by forcing continued re-encounters, at the cost of
departing from the paper's own unbounded assumption for this step. Step
3's `(Δr_o, Δr_a)` sweep above stays unbounded, since that is the direct
quantitative comparison against the paper's own Fig. 3(E)-(F).

**Known deviation: 2D `:dynamic_parallel` is more rigid than the paper's
region (c).** Every 2D point searched (a broad grid, multiple seeds, run
lengths up to 15000 steps) settled into one of exactly two stable regimes:
low polarisation (swarm/torus) or near-total polarisation (~0.95-0.999,
matching `:highly_parallel`), with no genuinely fluid, single-cluster,
intermediate-polarisation equilibrium found. The 2D preset used here
(`p_group≈0.92-0.94`) is the least-rigid *stable, single-cluster* point
found, not a confirmed match to Couzin's region (c); some apparent
"intermediate" points turned out to be two independently-rigid clusters
whose different headings only *look* like reduced polarisation in
aggregate (an artifact, not fluidity, checked directly via
connected-component analysis). The 3D preset below (B2.2) does show a
genuinely fluid, bounded, single-cluster intermediate regime, consistent
with Couzin (2002) being natively a 3D model: the 2D rules here are a
lower-dimensional adaptation, and this regime specifically appears not to
survive that reduction without further modification to the model.

In [ ]:
paper_regimes_2d = [:swarm, :milling, :dynamic_parallel, :highly_parallel]
regime_names_2d = Dict(
    :swarm            => "swarm",
    :milling          => "torus",
    :dynamic_parallel => "dynamic parallel group",
    :highly_parallel  => "highly parallel group",
)

trials_2d = Dict{Symbol, Any}()
for preset in paper_regimes_2d
    trial = generate_validated_preset(
        simulate_Couzin_2d, SimulationConfig(steps=5000, dt=0.1, seed=1), preset;
        tries=10, transient_steps=4000, L=150.0, param_kwargs=(N=100,),
        periodic=true, show_progress=false,
    )
    @assert trial.accepted "No seed within `tries` produced a $(preset) run passing preset_accepts."
    trials_2d[preset] = trial
    println(rpad(regime_names_2d[preset], 24), "seed=$(trial.seed) (trial $(trial.trial))  ",
        "p_group=$(round(trial.stats.polarisation_mean, digits=3))  ",
        "m_group=$(round(abs(trial.stats.rotation_mean), digits=3))")
end

#### B1.3 Parameter sweep: `(Δr_o, Δr_a)` phase structure in 2D

Couzin (2002) Fig. 3(E)-(F), group polarisation `p_group` and angular
momentum `m_group` as surfaces over `(Δr_o, Δr_a)`, is a **3D** result
(see B2.3 for the dimension-faithful reproduction). This section runs the
same sweep on the 2D adaptation instead: not a validation against the
paper, but worth seeing in its own right, since it's the version of the
model B1 actually iterates on, and the full sweep-and-ensemble machinery
below is exactly what B2.3 reuses on the 3D dynamics.

The sweep uses `simulate_Couzin2002_2d` (unbounded, `combine_rule=:couzin2002`)
with the paper's exact Table 1 parameters: `Couzin2002_base_params()`
already defaults to `N=100, α=270°, θ=40°/s, s=3, σ=0.05, τ=0.1`; only
`Zr/Zo/Za` change per grid point.

The grid starts at `Δr_o=Δr_a=0.1`, not `0`: the exact origin is Couzin's
own labelled fragmentation region (region e, ">50% chance of
fragmenting"), the single most failure-prone point in the whole grid. The
paper doesn't state an exact transient/averaging protocol beyond "analysed
after it reaches a dynamically stable state ... always within 5000 time
steps"; `transient_steps=4000` (discard the first 4000 of 5000 steps) is
one reasonable reading of that, not a verbatim paper value.

**Runtime**: the paper's own grid is 16x16 points x 30 replicates x 5000
steps x N=100, single-threaded; roughly 4 hours. Without the explicit
confirmation phrase, the cell instead runs a 5x5 x 3-replicate x
1500-step grid, about a minute,
enough to see the qualitative shape but not the full-resolution surface.
To run the full grid, type the exact confirmation phrase in the guarded
cell. A normal Run All always takes the quick branch.

In [ ]:
include("ABM/my_ABM_experiments.jl")   # parameter_combinations, needed by run_Couzin_phase_experiment

# Destructive-by-time guard: the full run starts only after typing the exact phrase.
full_scale_confirmation_2d = ""   # set to "RUN COUZIN 2D FULL SWEEP" deliberately
run_full_scale_2d = full_scale_confirmation_2d == "RUN COUZIN 2D FULL SWEEP"

params_paper_2d = Couzin2002_base_params()   # N=100, α=270°, θ=40°/s, s=3, σ=0.05, τ=0.1

if run_full_scale_2d
    Δro_vals_paper = collect(0.1:1.0:15.1)
    Δra_vals_paper = collect(0.1:1.0:15.1)
    nensemble_paper_2d = 30
    steps_paper_2d = 5000
    transient_paper_2d = 4000
else
    Δro_vals_paper = collect(0.1:3.5:15.1)
    Δra_vals_paper = collect(0.1:3.5:15.1)
    nensemble_paper_2d = 3
    steps_paper_2d = 1500
    transient_paper_2d = 1000
end
simcfg_paper_2d = SimulationConfig(steps=steps_paper_2d, dt=0.1, seed=1)

results_paper_2d, _ = run_Couzin_phase_experiment(
    simulate_Couzin2002_2d, simcfg_paper_2d, params_paper_2d;
    Δro_vals = Δro_vals_paper, Δra_vals = Δra_vals_paper, rr = 1.0,
    nensemble = nensemble_paper_2d, seed0 = 1, transient_steps = transient_paper_2d,
    collect_history = false, show_progress = true,
)
summary_paper_2d = summarise_Couzin_phase(results_paper_2d)
first(summary_paper_2d, 5)

In [ ]:
phase_paper_2d = Couzin_phase_plots(results_paper_2d;
    Δro_vals=Δro_vals_paper, Δra_vals=Δra_vals_paper)

using DelimitedFiles
couzin_2d_tag = run_full_scale_2d ? "full" : "quick"
couzin_2d_prefix = "FIGURES/ABM/couzin2002_2d_phase_$(couzin_2d_tag)"
writedlm("$(couzin_2d_prefix)_raw.csv", [permutedims(names(results_paper_2d)); Matrix(results_paper_2d)], ',')
writedlm("$(couzin_2d_prefix)_summary.csv", [permutedims(names(summary_paper_2d)); Matrix(summary_paper_2d)], ',')
save("$(couzin_2d_prefix)_heatmap.png", phase_paper_2d.fig_heatmap)
save("$(couzin_2d_prefix)_surface.png", phase_paper_2d.fig_surface)

display(phase_paper_2d.fig_heatmap)
display(phase_paper_2d.fig_surface)

This 2D phase surface is worth comparing to B2.3's 3D reproduction below
(same grid, same ensemble machinery), as a check on how much the 2D
adaptation's own phase structure resembles the native 3D dynamics, not
against Couzin's own Fig. 3 directly. See B2.3 below for the actual
paper comparison, figure included.

**Full-scale reproduction (pre-computed)**: the actual paper-scale grid
(16x16, 30 replicates, 5000 steps, N=100) run to completion through the
explicit confirmation branch
above, ~7.65 hours single-threaded. Not re-run live in this notebook,
same convention as Part C's Lymburn Fig. 8c reproduction; the exact code
above with the toggle flipped is what produced this.

![2D full-scale phase heatmap](../FIGURES/ABM/couzin2002_2d_phase_full_heatmap.png)

![2D full-scale phase surface](../FIGURES/ABM/couzin2002_2d_phase_full_surface.png)

Raw sweep tables are not distributed. To reproduce them, enter the exact
confirmation phrase in the guarded cell above and run it; the full 2D
sweep takes about 7.65 hours single-threaded. The cell saves newly generated
`_raw.csv` and `_summary.csv` files locally before displaying the figures.
Qualitatively: polarisation stays low across the whole
`Δr_o` range while `Δr_a` is large (low-`Δr_o` band), then rises as
`Δr_o` increases, consistent with the paper's own region layout; rotation
peaks in a localised band around moderate `Δr_o` (~10-15) and small `Δr_a`
(~2-5) rather than at small `Δr_o` with large `Δr_a` as sketched in
B1.3's qualitative description above, worth a direct look at the figure
rather than trusting that summary.

#### B1.4 Regime snapshots and animations

Panels (A)-(D) above are qualitative: swarm (loose, undirected), torus (a
rotating mill around an empty core), dynamic parallel group (aligned but
fluid: density and neighbours keep changing), highly parallel group
(aligned and rigid). The four validated runs from Step 2 already carry
full position/velocity history; reuse them directly for a snapshot grid
and one animation per regime, no re-simulation needed. These runs are
periodic (Step 2), so the fixed `[0,L]²` box is the correct view: no
re-centring needed.

In [ ]:
fig_2d_regimes = Figure(size=(1300, 380))
trail_len_regimes = 25
for (j, preset) in enumerate(paper_regimes_2d)
    trial = trials_2d[preset]
    out_r = trial.out
    L = trial.scenario.P.L
    k = length(out_r.pos_hist)

    # Naive `sum(pos)/length(pos)` + raw (unwrapped) positions breaks as
    # soon as a periodic group straddles the box edge: agents near x=0 and
    # x=L are *adjacent* in the periodic metric but average to a centre in
    # the middle of the box, and the raw spread then spans nearly the full
    # box width -- the "awful sizing" bug. periodic_centre_of_mass_2d +
    # displacement uses the same periodic convention in both cases
    # give the true periodic centre and the shortest wrapped offset from it.
    centre = periodic_centre_of_mass_2d(out_r.pos_hist[k], L)
    q = [displacement(centre, p, L) for p in out_r.pos_hist[k]]
    vel = out_r.vel_hist[k]

    ax = Axis(fig_2d_regimes[1, j], title=regime_names_2d[preset], xlabel="x", ylabel="y", aspect=DataAspect())

    # Short recent-trajectory trail, recentred on the SAME fixed `centre` as
    # the final frame (not re-centred per frame) so the trail stays visually
    # continuous instead of jumping every time the periodic centre itself
    # is recomputed.
    k0 = max(1, k - trail_len_regimes + 1)
    for i in eachindex(out_r.pos_hist[k])
        trail_pts = [displacement(centre, out_r.pos_hist[f][i], L) for f in k0:k]
        lines!(ax, getindex.(trail_pts, 1), getindex.(trail_pts, 2); color=(:grey40, 0.35), linewidth=1.0)
    end

    ap, ad = arrows_from(q, vel; shaft_len=0.9)
    arrows2d!(ax, ap, ad; tiplength=8, shaftwidth=1.5, color=:dodgerblue)
    scatter!(ax, getindex.(q, 1), getindex.(q, 2); markersize=4, color=:dodgerblue)
end
save("FIGURES/ABM/couzin2002_2d_regimes.png", fig_2d_regimes)
fig_2d_regimes

In [ ]:
generate_validation_animations = false  # set true to regenerate optional Couzin videos
if generate_validation_animations
    for preset in paper_regimes_2d
        animcfg_r = AnimationConfig(
            save_mp4   = true,
            filename   = "ANIMATIONS/ABM/couzin2002_2d_$(preset).mp4",
            stride     = 5,
            fps        = 30,
            show_zones = false,
            highlight_focal = false,
        )
        animate_simulation_2d(trials_2d[preset].out, animcfg_r; title = "Couzin 2002: $(regime_names_2d[preset])")
    end
else
    println("Skipped optional 2D Couzin animations; set generate_validation_animations=true to regenerate them.")
end

#### B1.5 Collective-order diagnostics

The four order parameters recorded automatically during simulation
(dilation, rotation, polarisation, absolute angular momentum), and their
means after discarding an initial transient. Applied here to the small
periodic demo run from Step 1.

In [ ]:
plot_order_parameters(out_b)

In [ ]:
mop = mean_order_parameters(out_b; transient_steps=50)
println("rotation (mean):     ", mop.rotation_mean)
println("polarisation (mean): ", mop.polarisation_mean)

#### B1.6 Interaction-network topology

Treat the swarm as a time-varying interaction graph: who is currently
influencing whom, from the same repulsion/orientation/attraction rules
that drive the dynamics, and track that graph's structure over time
(`NETWORKS/my_networks.jl`, `ABM/models/Couzin/my_Couzin_networks.jl`).

In [ ]:
include("NETWORKS/my_networks.jl")

graphs_b = Couzin_active_graph_series_2d(out_b, P_b)   # one directed graph per stored frame

localdf  = local_stats_time_series(graphs_b; t=out_b.t)    # per-agent, per-frame
globaldf = global_stats_time_series(graphs_b; t=out_b.t)   # one row per frame
first(globaldf, 3)

In [ ]:
fig_net = Figure(size=(900, 650))
ax1 = Axis(fig_net[1, 1], ylabel="density", title="Interaction-network statistics over time")
ax2 = Axis(fig_net[2, 1], ylabel="mean clustering")
ax3 = Axis(fig_net[3, 1], ylabel="giant component", xlabel="t")
lines!(ax1, globaldf.t, globaldf.density)
lines!(ax2, globaldf.t, globaldf.mean_clustering)
lines!(ax3, globaldf.t, globaldf.giant_component)
fig_net

Cluster (connected-component) structure: how many separate clusters exist at each frame, and how polarised is each one on average?

In [ ]:
cluster_df = cluster_order_parameters_time_series(
    graphs_b, out_b.pos_hist, out_b.vel_hist, P_b.L;
    displacement = displacement, t = out_b.t,
)
cluster_summary = summarise_cluster_order_parameters(cluster_df)

fig_clust = Figure(size=(900, 400))
ax1 = Axis(fig_clust[1, 1], ylabel="# clusters", xlabel="t", title="Cluster count over time")
ax2 = Axis(fig_clust[1, 2], ylabel="mean polarisation (weighted by cluster size)", xlabel="t", title="Cluster-level polarisation")
lines!(ax1, cluster_summary.t, cluster_summary.n_clusters)
lines!(ax2, cluster_summary.t, cluster_summary.polarisation_weighted_mean)
fig_clust

### B2. Three-dimensional Couzin model

Couzin et al. (2002) formulated this model in **unbounded three-dimensional
space**; B1 above is a periodic 2D adaptation useful for fast iteration,
not the paper's own geometry. This section reproduces the model as
published: spherical repulsion/orientation/attraction zones, a 90° blind
cone behind each agent (270° field of perception), finite turning speed,
constant forward speed, spherically distributed angular error, and an
unbounded domain (`periodic=false`).

Reference: Couzin, Krause, James, Ruxton & Franks (2002), *Collective
Memory and Spatial Sorting in Animal Groups*, J. Theor. Biol. 218, 1-11,
doi:10.1006/jtbi.2002.3065.

#### B2.1 Construct and simulate

In [ ]:
# Standalone includes: this section does not depend on B1 having run first.
include("ABM/load_ABM.jl")
include("ABM/models/Couzin/load_Couzin.jl")

# Same four regimes/labels as B1 (redefined here so B2 stays runnable on its own).
paper_regimes_2d = [:swarm, :milling, :dynamic_parallel, :highly_parallel]
regime_names_2d = Dict(
    :swarm            => "swarm",
    :milling          => "torus",
    :dynamic_parallel => "dynamic parallel group",
    :highly_parallel  => "highly parallel group",
)

scenario_3d = Couzin_params_from_preset(:milling; dim=3, N=50, L=50.0, dt=0.1)
simcfg_3d_demo = SimulationConfig(steps=1000, dt=scenario_3d.dt, seed=1)

out_3d_demo = simulate_Couzin_3d(simcfg_3d_demo, scenario_3d.P)
(length(out_3d_demo.pos_hist), length(out_3d_demo.t))

#### B2.2 Validated instances of Couzin's four named regimes

Same four regimes and same `generate_validated_preset`/`preset_accepts`
path as B1, now via `Couzin_params_from_preset(...; dim=3)`. 3D needs
different `(Zo, Za)` than 2D for the same qualitative regime: a spherical
orientation zone holds far more neighbours than the equivalent disc at the
same radius, and the blind cone blocks a smaller fraction of a sphere than
of a circle, so 2D-tuned values over-align in 3D. `COUZIN_3D_OVERRIDES`
(`my_Couzin_core.jl`) holds the independently re-validated `(Zo, Za)` per
regime, applied automatically by `dim=3`.

**Periodic, not unbounded**: same reasoning as B1 Step 2. An unbounded
`:dynamic_parallel` disperses (checked directly via group radius) rather
than settling, so it, and, for consistency, the other three regimes too,
are validated here under `simulate_Couzin_3d` with `periodic=true`.
Step 3's sweep below stays unbounded, matching the paper's own geometry
for the direct Fig. 3 comparison.

In [ ]:
trials_3d = Dict{Symbol, Any}()
for preset in paper_regimes_2d
    trial = generate_validated_preset(
        simulate_Couzin_3d, SimulationConfig(steps=5000, dt=0.1, seed=1), preset;
        tries=10, transient_steps=4000, L=75.0, param_kwargs=(N=100, dim=3),
        periodic=true, show_progress=false,
    )
    @assert trial.accepted "No seed within `tries` produced a $(preset) run passing preset_accepts."
    trials_3d[preset] = trial
    println(rpad(regime_names_2d[preset], 24), "seed=$(trial.seed) (trial $(trial.trial))  ",
        "p_group=$(round(trial.stats.polarisation_mean, digits=3))  ",
        "m_group=$(round(abs(trial.stats.rotation_mean), digits=3))")
end

#### B2.3 Reproducing Couzin (2002), Fig. 3 (the 3D model)

The four validated points above confirm the named regimes individually;
this sweep reproduces the continuous `p_group`/`m_group` surfaces, the
paper's own result, since Couzin (2002) is explicitly three-dimensional
("simulated ... in continuous three-dimensional space"). B1's 2D sweep is
a faster, lower-dimensional adaptation of the same rules; this is the
dimension-faithful comparison. Same grid, replicate count and transient
convention as B1.

**Runtime**: same profile as B1's sweep, plausibly more expensive since a
3D neighbour search touches one more dimension per pairwise check.
A normal execution below runs the same fast 5x5 x 3-replicate check as
B1 (a few minutes). The paper-scale 16x16 x 30-replicate branch requires
typing its exact confirmation phrase and is likely 6+ hours.

In [ ]:
include("ABM/my_ABM_experiments.jl")   # parameter_combinations, needed by run_Couzin_phase_experiment

# Destructive-by-time guard: the full run starts only after typing the exact phrase.
full_scale_confirmation_3d = ""   # set to "RUN COUZIN 3D FULL SWEEP" deliberately
run_full_scale_3d = full_scale_confirmation_3d == "RUN COUZIN 3D FULL SWEEP"

params_paper_3d = Couzin2002_base_params()

if run_full_scale_3d
    Δro_vals_paper_3d = collect(0.1:1.0:15.1)
    Δra_vals_paper_3d = collect(0.1:1.0:15.1)
    nensemble_paper_3d = 30
    steps_paper_3d = 5000
    transient_paper_3d = 4000
else
    Δro_vals_paper_3d = collect(0.1:3.5:15.1)
    Δra_vals_paper_3d = collect(0.1:3.5:15.1)
    nensemble_paper_3d = 3
    steps_paper_3d = 1500
    transient_paper_3d = 1000
end
simcfg_paper_3d = SimulationConfig(steps=steps_paper_3d, dt=0.1, seed=1)

results_paper_3d, _ = run_Couzin_phase_experiment(
    simulate_Couzin2002_3d, simcfg_paper_3d, params_paper_3d;
    Δro_vals = Δro_vals_paper_3d, Δra_vals = Δra_vals_paper_3d, rr = 1.0,
    nensemble = nensemble_paper_3d, seed0 = 1, transient_steps = transient_paper_3d,
    collect_history = false, show_progress = true,
)
summary_paper_3d = summarise_Couzin_phase(results_paper_3d)
first(summary_paper_3d, 5)

In [ ]:
phase_paper_3d = Couzin_phase_plots(results_paper_3d;
    Δro_vals=Δro_vals_paper_3d, Δra_vals=Δra_vals_paper_3d)

using DelimitedFiles
couzin_3d_tag = run_full_scale_3d ? "full" : "quick"
couzin_3d_prefix = "FIGURES/ABM/couzin2002_3d_phase_$(couzin_3d_tag)"
writedlm("$(couzin_3d_prefix)_raw.csv", [permutedims(names(results_paper_3d)); Matrix(results_paper_3d)], ',')
writedlm("$(couzin_3d_prefix)_summary.csv", [permutedims(names(summary_paper_3d)); Matrix(summary_paper_3d)], ',')
save("$(couzin_3d_prefix)_heatmap.png", phase_paper_3d.fig_heatmap)
save("$(couzin_3d_prefix)_surface.png", phase_paper_3d.fig_surface)

display(phase_paper_3d.fig_heatmap)
display(phase_paper_3d.fig_surface)

Compare directly against Couzin (2002) Fig. 3(E)-(F) below: `p_group`
should stay low across the whole `Δr_o` range while `Δr_a` is large
(torus, region b), then rise sharply as `Δr_o` increases (parallel
groups, regions c-d); `m_group` should peak in a ridge at small `Δr_o` +
large `Δr_a`, then fall as `Δr_o` increases.

See Fig. 3 in [Couzin et al. (2002)](https://doi.org/10.1006/jtbi.2002.3065). The source figure is not redistributed with this repository.

**Open discrepancy, not yet resolved**: both this notebook's own
full-scale reproductions (2D below, and this section's own 3D one)
consistently put the `m_group`/rotation peak at moderate-to-large `Δr_o`
and *small* `Δr_a` instead, the opposite corner from what's described
above. That description was read directly off the paper's own Fig. 3(F)
during earlier work on this notebook; either that reading is wrong, or
there's a real deviation between this implementation and the paper's own
sweep somewhere. Two independent full-scale runs (2D and 3D, different
code paths, same qualitative disagreement) rules out "one-off simulation
fluke" as the explanation; worth checking against the paper's actual
Fig. 3(F) image above directly rather than trusting either description
before digging further.

Also worth checking against B1's own 2D sweep (`phase_paper_2d`): the two
should agree qualitatively (same regions, same rough shape) if the 2D
adaptation is a reasonable stand-in for the paper's native 3D dynamics; a
material disagreement is itself a useful finding about how much the
dimensional reduction changes the model's behaviour.

**Full-scale reproduction (pre-computed)**: the actual paper-scale grid
(16x16, 30 replicates, 5000 steps, N=100, 3D) run to completion through
the explicit confirmation branch, ~8.3 hours single-threaded (slightly
longer than B1's 2D version at the same replicate count, consistent with
the 3D neighbour search being more expensive per step). Not re-run live
in this notebook, same convention as Part C's Lymburn Fig. 8c
reproduction; the exact code above with the toggle flipped is what
produced this.

![3D full-scale phase heatmap](../FIGURES/ABM/couzin2002_3d_phase_full_heatmap.png)

![3D full-scale phase surface](../FIGURES/ABM/couzin2002_3d_phase_full_surface.png)

Raw sweep tables are not distributed. To reproduce them, enter the exact
confirmation phrase in the guarded cell above and run it; the full 3D
sweep takes about 8.3 hours single-threaded. The cell saves newly generated
`_raw.csv` and `_summary.csv` files locally. Qualitatively, this
is a cleaner separation than B1's 2D version: polarisation rises sharply
once `Δr_o` exceeds ~3-5 and stays near 1 across almost the whole
remaining grid, low only in the small-`Δr_o`/small-`Δr_a` corner; rotation
peaks in a clear band at `Δr_o` roughly 7+ and `Δr_a` below ~4, the same
qualitative "moderate-to-large `Δr_o`, small `Δr_a`" region B1's 2D full
run also found, not the small-`Δr_o`/large-`Δr_a` region the earlier
qualitative sketch in B1.3/B2.3 described. Worth updating that
description rather than the figures if the two keep disagreeing after
independent runs.

#### B2.4 Regime snapshots and animations

Same idea as B1 Step 4, in 3D: reuse the validated (periodic) runs from
Step 2 (`trials_3d`, already have full history) for a snapshot grid and
one animation per regime. Fixed `[0,L]³` box, no re-centring needed.

In [ ]:
fig_3d_regimes = Figure(size=(1100, 300))
for (j, preset) in enumerate(paper_regimes_2d)
    trial = trials_3d[preset]
    out_r = trial.out
    L = trial.scenario.P.L
    pos = out_r.pos_hist[end]

    # Same periodic-centring fix as the 2D regimes figure above: a naive
    # arithmetic-mean centre and raw (unwrapped) positions break once a
    # periodic group straddles the box edge.
    centre = periodic_centre_of_mass_3d(pos, L)
    q = [displacement(centre, p, L) for p in pos]

    ax = Axis3(fig_3d_regimes[1, j], title=regime_names_2d[preset],
        xlabel="x", ylabel="y", zlabel="z", aspect=(1, 1, 1))
    scatter!(ax, getindex.(q, 1), getindex.(q, 2), getindex.(q, 3);
        markersize=7, color=getindex.(q, 3), colormap=:viridis)
end
save("FIGURES/ABM/couzin2002_3d_regimes.png", fig_3d_regimes)
fig_3d_regimes

In [ ]:
if generate_validation_animations
    for preset in paper_regimes_2d
        animcfg_3d = AnimationConfig(
            save_mp4    = true,
            filename    = "ANIMATIONS/ABM/couzin2002_3d_$(preset).mp4",
            stride      = 5,
            fps         = 30,
            show_zones  = false,
            highlight_focal = false,
            fig_size    = (700, 700),
        )
        animate_simulation_3d(trials_3d[preset].out, animcfg_3d;
            title = "Couzin 2002: $(regime_names_2d[preset])")
    end
else
    println("Skipped optional 3D Couzin animations; set generate_validation_animations=true to regenerate them.")
end

#### B2.5 Collective-order and fluidity diagnostics

`p_group`/`m_group` alone (already checked via `preset_accepts` in Step 2)
can't distinguish a fluid dynamic-parallel flock from a rigid
highly-parallel one: both have high polarisation and low rotation. The
dynamic case should show substantially more group-size fluctuation,
pair-distance change and neighbour turnover than the highly-parallel
control.

In [ ]:
function Couzin_fluidity_3d(out, P; first=length(out.pos_hist) - 599, lag=100)
    disp = displacement
    pairdist(pos) = [norm(disp(pos[i], pos[j], P.L))
        for i in 1:length(pos)-1 for j in i+1:length(pos)]
    function kneighbours(pos, k=5)
        N = length(pos)
        [Set(sortperm([i == j ? Inf : norm(disp(pos[i], pos[j], P.L))
            for j in 1:N])[1:k]) for i in 1:N]
    end
    radius = Float64[]
    for t in first:10:length(out.pos_hist)
        pos = out.pos_hist[t]
        centre = periodic_centre_of_mass_3d(pos, P.L)
        push!(radius, sqrt(mean(norm(disp(centre, p, P.L))^2 for p in pos)))
    end
    pairchange, turnover = Float64[], Float64[]
    for t in first:lag:(length(out.pos_hist)-lag)
        p0, p1 = out.pos_hist[t], out.pos_hist[t+lag]
        d0, d1 = pairdist(p0), pairdist(p1)
        push!(pairchange, mean(abs.(d1 .- d0)) / mean(d0))
        n0, n1 = kneighbours(p0), kneighbours(p1)
        push!(turnover, mean(1 - length(intersect(n0[i], n1[i])) /
            length(union(n0[i], n1[i])) for i in eachindex(n0)))
    end
    return (radius_cv=std(radius)/mean(radius),
        pair_distance_change=mean(pairchange), neighbour_turnover=mean(turnover))
end

fluidity_3d = Dict(name => Couzin_fluidity_3d(trials_3d[name].out, trials_3d[name].scenario.P)
    for name in (:dynamic_parallel, :highly_parallel))

println("dynamic fluidity:       ", fluidity_3d[:dynamic_parallel])
println("highly-parallel control:", fluidity_3d[:highly_parallel])

fd, fh = fluidity_3d[:dynamic_parallel], fluidity_3d[:highly_parallel]
@assert fd.radius_cv > 1.5fh.radius_cv
@assert fd.pair_distance_change > 1.5fh.pair_distance_change
@assert fd.neighbour_turnover > 1.5fh.neighbour_turnover

#### B2.6 Interaction-network topology

Repeat B1's network analysis on the validated 3D dynamic-parallel run. The
graph construction uses the same (periodic) displacement rule as the
simulation.

In [ ]:
include("NETWORKS/my_networks.jl")

out_network_3d = trials_3d[:dynamic_parallel].out
P_network_3d = trials_3d[:dynamic_parallel].scenario.P
t_idxs_network_3d = unique(round.(Int, range(1, length(out_network_3d.pos_hist); length=40)))
graphs_3d = [interaction_layers(
    out_network_3d.pos_hist[t], out_network_3d.vel_hist[t], P_network_3d;
    displacement_fn=displacement,
).active for t in t_idxs_network_3d]
globaldf_3d = global_stats_time_series(graphs_3d; t=out_network_3d.t[t_idxs_network_3d])

fig_net_3d = Figure(size=(900, 650))
ax1 = Axis(fig_net_3d[1, 1], ylabel="density", title="3D interaction-network statistics")
ax2 = Axis(fig_net_3d[2, 1], ylabel="mean clustering")
ax3 = Axis(fig_net_3d[3, 1], ylabel="giant component", xlabel="t")
lines!(ax1, globaldf_3d.t, globaldf_3d.density)
lines!(ax2, globaldf_3d.t, globaldf_3d.mean_clustering)
lines!(ax3, globaldf_3d.t, globaldf_3d.giant_component)
fig_net_3d

#### B2.7 Preparing the 3D state for a reservoir comparison

Flattening 3D should vectorise the full state, not discard z. The first
matrix has one column per time point and 6N rows: centred (x,y,z)
positions then (vx,vy,vz) velocities. The second is an explicit xy
projection with 4N rows. Keeping both makes a later 2D-vs-3D reservoir
comparison auditable.

In [ ]:
function Couzin_state_matrix_3d(out; projection=:xyz, include_velocity=true)
    @assert projection in (:xyz, :xy)
    dims = projection == :xyz ? (1, 2, 3) : (1, 2)
    N, T = length(out.pos_hist[1]), length(out.pos_hist)
    nblocks = include_velocity ? 2 : 1
    X = Matrix{Float64}(undef, nblocks * length(dims) * N, T)
    for t in 1:T
        centre = sum(out.pos_hist[t]) / N
        centred = out.pos_hist[t] .- Ref(centre)
        p = reduce(vcat, ([x[d] for d in dims] for x in centred))
        if include_velocity
            v = reduce(vcat, ([x[d] for d in dims] for x in out.vel_hist[t]))
            X[:, t] = vcat(p, v)
        else
            X[:, t] = p
        end
    end
    return X
end

out_compare_3d = trials_3d[:dynamic_parallel].out
X3_flat = Couzin_state_matrix_3d(out_compare_3d; projection=:xyz)
X3_xy   = Couzin_state_matrix_3d(out_compare_3d; projection=:xy)
println("full centred 3D state: ", size(X3_flat))
println("explicit xy projection: ", size(X3_xy))

For the later model decision, compare the existing 2D state, `X3_xy`, and
`X3_flat` under the same input, train/test split, readout regularisation
and random seeds. Because `X3_flat` has more features, report both raw
performance and a dimension-controlled comparison (e.g. PCA to a common
feature count). Don't choose between them from a single attractive
trajectory.

## C. The swarm itself as a reservoir

`SWARM_RC/` wires a swarm and the reservoir-training pipeline (Part A)
together: the swarm's *own* agent dynamics play the role the random
recurrent network played for the ESN. 

Two swarm models are available here:

- **Lymburn et al. (2021)**: the model the swarm-reservoir approach was
  originally published with (*"Reservoir Computing with Swarms"*, Chaos
  2021). Featured first below because it's the paper-faithful
  reference point.
- **Couzin**: the alternative model already built out in Part B. It
  works as a swarm-reservoir too (shown second, below), but getting good
  performance out of it takes more careful tuning than the Lymburn model
  needs out of the box.

### Lymburn et al. (2021): the original swarm-reservoir model

Lymburn, Algar, Small & Jüngling (2021), *Reservoir computing with
swarms*, Chaos 31, 033121. [doi:10.1063/5.0039745](https://doi.org/10.1063/5.0039745).

A modified Reynolds-boids flock: no periodic domain. A global *homing*
force to a fixed point holds the swarm together instead of a box.
Repulsion (Eq 1) is continuous and distance-weighted (`1/r²`) within a
hard cutoff radius `rr`; alignment (Eq 2) is Vicsek-style (velocity-based
rather than position-based), but a *flat* sum of neighbours' velocity
differences with no distance-weighting at all, within its own hard
cutoff radius `ra`. So the two neighbour forces are a bit of a hybrid:
both use a hard-cutoff neighbourhood (unlike Couzin's concentric zones
elsewhere in this codebase), but only repulsion falls off continuously
with distance inside that cutoff. Alignment treats a neighbour at the
edge of `ra` exactly the same as one right next to the agent. The
predator enters as a literal 5th force term (Eq 10 of the paper) rather
than steering a "desired direction". The Gaussian-kernel observation
layer (`build_observation_layer_lymburn!`) is the same algorithm either
way: it's literally named after this paper.

In [ ]:
include("SWARM_RC/load_SwarmRC_Lymburn.jl")   # loads ABM + RC + Couzin + Lymburn + SWARM_RC in the right order
include("ABM/my_ABM_experiments.jl")           # parameter_combinations/summarise_experiment: needed by run_Lymburn_rc_sweep below, even if running Part C standalone
include("RC/my_consistency.jl")
include("RC/my_reservoir_report.jl")              # ridge/conditioning diagnostics; keeps Part C standalone
include("TIME_SERIES/my_systems.jl")
include("SWARM_RC/my_predator.jl");                # input signal scaling/coupling helpers

### C0. Build the slide in code: three choices, then one trained readout

A swarm reservoir is not just “a swarm used as an ESN”. It is a modelling
pipeline with three independently named choices:

| component | mathematical role | choice in the paper-faithful demo | alternatives already in this codebase |
|---|---|---|---|
| **Input and coupling** | signal $u(n)$ and the rule by which it perturbs the agents | rescaled Lorenz $(x,y)$ as a predator | no coupling, or a new subtype of `InputCoupling` |
| **Swarm dynamics** | physical state update $\mathbf{x}(n+1)=g(\mathbf{x}(n),u(n))$ | Lymburn/Reynolds swarm | Couzin zonal swarm; parameters and dimensionality are also choices |
| **Observation layer** | what is made visible, $\mathbf{z}(n)=\phi(\mathbf{x}(n))$ | Gaussian spatial kernels | raw positions; the same spatial kernels work for either 2D swarm, with coverage/k-means as an alternative placement rule |

The target and forecast horizon define the **task**, not the reservoir.
Only the final linear readout is fitted. Keeping the input *signal* separate
from its *coupling rule* matters: swapping Lorenz for another time series is
not automatically meaningful unless its units, scale and sampling time are
adapted to the chosen swarm.

The following object is the talk's switchboard. It makes every box on the
slide explicit before any simulation is run.

In [ ]:
swarm_pipeline = (
    input = (
        signal = :lorenz_xy,
        coupling = :predator_force,
        transform = :zscore,
        target_std = 2.0,
    ),
    dynamics = (
        model = :lymburn,
        preset = :critical,
        N = 200,
        dt = 0.02,
    ),
    observation = (
        compare = (:raw_positions, :spatial_gaussian),
        selected = :spatial_gaussian,
        M = 200,
        kneigh = 5,
    ),
    task = (
        target = :future_input_x,
        horizon_time = 0.5,
    ),
)

swarm_pipeline

**What students should notice.** A new swarm model implements the common
`reset!`, `reservoir_step!` and `raw_state` interface. A new observation
implements `feature_map`; a stateful one also implements
`build_observation_layer!`. A new way of injecting a signal implements
`InputCoupling`. The generic feature collection and ridge-readout code then
stays unchanged. Below we first expose the physical state (positions and
velocities), then compare two observation maps of the *same driven
trajectory*. This is the cleanest demonstration that the observation layer
is a modelling choice rather than the swarm itself.

> **Important:** `swarm_pipeline` is an explicit record of the choices, not
> yet a universal factory. Changing a symbol is only safe when the
> compatibility table below says the combination is implemented.

#### What is actually switchable today?

| component | **Lymburn** | **Mizzi** | **Lund** |
|---|---|---|---|
| **Swarm** | 2D interacting swarm with repulsion, alignment, homing and friction | 2D territorial agents with fixed homes, homing, prey response and friction | 2D toroidal swarm with hysteretic dispersed and clustered states |
| **Input** | Lorenz $(x,y)$ projection | scalar Lorenz delay embedding $(u(t-\tau),u(t))$ | standardised scalar Lorenz signal |
| **Coupling** | moving-predator force or `NoCoupling` | moving-prey force or `NoCoupling` | global temperature-to-target-speed coupling or `NoCoupling` |
| **Observation** | $3M$ Gaussian density and velocity responses; raw $2N$ positions are available for comparison | direct $4N$ home-relative positions and velocities | nine state-aware aggregate measurements |
| **Tutorial target** | future Lorenz $x$, teacher-forced | next scalar sample, teacher-forced | future scalar sample, teacher-forced |
| **Main limitation** | the tutorial force cap and training task differ from the paper | MDL optimisation and the paper's autonomous evaluation are omitted | reduced to $N=200$ and used for prediction rather than reproducing the paper's full tests |

All three use the same affine ridge readout. `validate_pipeline` checks that the selected coupling and observation are supported, dimensions and data are finite, input scale and timestep are plausible, observations vary and have usable rank, and the target matches the prediction mode. `NoCoupling` is a control; other component types are extension points, not ready-made interchangeable choices.

#### Compatibility and adequacy checks

Checks should be made at three different levels:

1. **Can these components be connected?** This is partly enforced now:
   two-dimensional swarm input, supported observation method, valid kernel
   parameters, and matching feature/target lengths. These are hard pass/fail
   checks.
2. **Are their physical scales compatible?** Utilities exist, but this is
   not automatically enforced. `match_predator_to_couzin` and
   `match_predator_to_lymburn` match spatial extent, speed and timestep;
   `print_matching_report` reports the resulting ratios, and Couzin has
   nondimensionalisation audits. The paper-faithful Lymburn path deliberately
   uses its published per-coordinate standard-deviation scaling instead.
   A user can still bypass all of these and drive a swarm with a badly scaled
   signal.
3. **Is the resulting reservoir computationally useful?** The code provides
   feature conditioning/effective-rank diagnostics, memory capacity,
   perturbation stability, consistency, sequence separability, held-out task
   error/correlation, and swarm response/order parameters. These are
   available diagnostics, not an automatic acceptance test, and the Tutorial
   should interpret them together rather than declare one universal threshold.

`validate_pipeline` now combines these checks into one pre-training report.
It rejects structural incompatibilities, audits supplied scale matching,
probes observation constancy/effective rank, and distinguishes warnings
from errors. It does not silently rescale an input: the chosen adapter is a
scientific modelling decision and must remain visible.

**About 3D Couzin.** The codebase has a 3D Couzin ABM, but it cannot simply
be passed to the present `CouzinReservoir`, whose state, coupling, raw-state
conversion, perturbation and Gaussian feature methods are all explicitly
2D. A genuine 3D reservoir needs a 3-vector coupling, 3D periodic distance,
3D kernel centres and density plus three velocity-weighted channels
($4M$ features rather than $3M$). That is straightforward conceptually but
large enough to deserve a separate advanced 3D swarm-reservoir section and
validation path, rather than a misleading `dim=3` switch in this first demo.

Qualitative look: same idea as Part B's snapshot, but note there's no
domain box: the swarm just stays near its home point (the origin) under
its own homing force. Colour = each agent's **heading** (`atan(vy, vx)`)
on a cyclic colourmap; the trailing path behind each agent (last 25
steps) makes swirling/milling motion visible in a single static frame.

In [ ]:
P_demo = Lymburn_params_from_preset(:critical; N=200)   # Kr=2, Ka=0.01: the paper's own "point B", best-performing regime
simcfg_demo = SimulationConfig(steps=6000, dt=0.02, seed=1)   # long enough to see the swarm actually organise (see below)
out_lymburn = simulate_Lymburn_2d(simcfg_demo, P_demo; show_progress=false)

# Front-load the snapshots so the initial organisation transient is visible.
snapshot_times_lym = [0.0, 5.0, 7.0, 10.0]
frames_to_show = [argmin(abs.(out_lymburn.t .- ts)) for ts in snapshot_times_lym]
fig_lym = Figure(size=(1000, 260))
for (i, k) in enumerate(frames_to_show)
    ax = Axis(fig_lym[1, i], title = "t = $(round(out_lymburn.t[k], digits=1))  (Φ_R=$(round(out_lymburn.rotation[k], digits=2)))", aspect = DataAspect())
    plot_swarm_frame!(ax, out_lymburn.pos_hist, out_lymburn.vel_hist, k; trail_len=25, agent_ms=6.0)
    xlims!(ax, -10, 10)
    ylims!(ax, -10, 10)
end
Colorbar(fig_lym[1, 5], limits=(-pi, pi), colormap=:hsv, label="heading (rad)",
    ticks=([-pi, -pi/2, 0, pi/2, pi], ["-π", "-π/2", "0", "π/2", "π"]))

save("FIGURES/swarmRC/lymburn_swarm_snapshot.png", fig_lym)
fig_lym

> **Implementation note:** exact friction/force-cap choices and the
> paper-faithful versus footprint-matched input-scaling trade-off are kept
> in `TECHNICAL_NOTES.md`. They are not needed to use the starter
> pipeline.

### The paper's actual task

The paper drives the swarm with the Lorenz system, rescaled to standard
deviation 2 in each dimension (`zscore_rescale`), sampled uniformly at
`dt=0.02` (matching the swarm's own step size), then trains a readout to
predict the Lorenz x-coordinate **0.5 time units ahead**: a genuine
forecasting task, not "predict the next input sample". That's different
from how `train_and_evaluate_reservoir` is used elsewhere in this
notebook, so this section builds the training pipeline by hand.

This is deliberately a **teacher-forced scalar forecast**, not a free-run
generator: the true two-coordinate predator trajectory continues to drive
the swarm while the readout estimates future $x$. Feeding that single
number back into a reservoir that requires a two-coordinate input would be
dimensionally and conceptually wrong. An autonomous version must instead
predict both next predator coordinates, or use a scalar delay embedding and
predict/update the complete next embedding state. `validate_pipeline`
rejects the ambiguous scalar-output/2D-input free-run combination.

In [ ]:
# 1. INPUT and 2. SWARM DYNAMICS: instantiate the choices declared above.
dt = swarm_pipeline.dynamics.dt
P = Lymburn_params_from_preset(swarm_pipeline.dynamics.preset; N=swarm_pipeline.dynamics.N)   # paper's point B
res_lym = build_Lymburn_reservoir(P, dt; rng=MersenneTwister(1))

rng_lorenz = MersenneTwister(7)
lor = lorenz_data(rng = rng_lorenz, tspan = (0.0, 120.0), dtmax = 0.01)
tgrid = collect(0.0:dt:120.0)
X3 = Array(lor.sol(tgrid))              # uniform dt=0.02 samples, unlike lor.data (raw adaptive steps)

U_full = zscore_rescale(X3[1:2, :]; target_std=swarm_pipeline.input.target_std)   # input adapter

horizon = round(Int, swarm_pipeline.task.horizon_time / dt)   # 25 steps = 0.5 time units
Ttot = size(U_full, 2) - horizon
U = U_full[:, 1:Ttot]
target = U_full[1, (1:Ttot) .+ horizon]   # z(t) = rescaled predator x(t + 0.5), matching the paper's own x_p(1,tau+1:end)

size(U), length(target)

### Naive particle positions vs. the Gaussian-kernel observation layer

This is the paper's headline result (Figs 2 and 5): using the swarm's raw
`2N` particle-position coordinates directly as the reservoir state performs
poorly, because agents are interchangeable (permutation symmetry): two
different agents occupying each other's positions look identical to a
linear readout, but drive the *individual* trajectories to diverge across
replicas. The Gaussian-kernel observation layer sidesteps this by measuring
the swarm's *shape*, which is permutation-invariant.

In [ ]:
shift, train_len, predict_len, washout = 200, 3000, 800, 200

# 3. OBSERVATION: fit the selected spatial map. `nothing` below selects raw positions.
K_lym = build_observation_layer!(res_lym, U[:, 1:shift+train_len];
    washout=washout, rng=MersenneTwister(2), method=:spatial_gaussian,
    M=swarm_pipeline.observation.M, kneigh=swarm_pipeline.observation.kneigh, show_progress=false)

pipeline_check = validate_pipeline(res_lym, U;
    observation=:spatial_gaussian, obs=K_lym,
    target=target, prediction_mode=:teacher_forced,
    dt_input=dt, probe_steps=300, rng=MersenneTwister(90))
print_pipeline_validation(pipeline_check)
pipeline_check.valid || error("Invalid swarm-reservoir pipeline; see report above.")

function lymburn_collect(res, obs)
    reset!(res; rng=MersenneTwister(1))
    X, _ = collect_features(res, U[:, 1:shift+train_len+predict_len];
        rng=MersenneTwister(3), log_raw=false, reset_res=true,
        feature_fn=feature_map, obs=obs, show_progress=false)
    return X
end

function lymburn_fit_R(Xfeat; ridge_grid)
    Y = reshape(target[1:shift+train_len+predict_len], 1, :)
    Xtr = Xfeat[:, shift+washout+1:shift+train_len]
    Ytr = Y[:, shift+washout+1:shift+train_len]
    Xte = Xfeat[:, shift+train_len+1:shift+train_len+predict_len]
    Yte = Y[:, shift+train_len+1:shift+train_len+predict_len]
    Wout, μx, σx, best_λ, _ = train_readout_cv(Xtr, Ytr; ridge_grid=ridge_grid, n_folds=5, gap=25)
    Yhat = apply_readout(Wout, Xte, μx, σx)
    return cor(vec(Yhat), vec(Yte)), Yhat, Yte, best_λ
end

# naive's raw 400-dim positions are far more ill-conditioned than kernel's ~600-dim
# Gaussian features, so it needs a much wider ridge search. λ selection is via
# held-out MSE, matching the paper's own CVALridge.m recipe in spirit.
R_naive, Yhat_naive, Yte_naive, λ_naive = lymburn_fit_R(lymburn_collect(res_lym, nothing); ridge_grid=10.0 .^ (-2:1:6))
R_kernel, Yhat_kernel, Yte_kernel, λ_kernel = lymburn_fit_R(lymburn_collect(res_lym, K_lym); ridge_grid=10.0 .^ (-3:1:5))

println("naive  (raw 2N positions): R = ", round(R_naive, digits=3), "  (λ=", λ_naive, ")   (paper reports R ≈ 0.19)")
println("kernel (M=200 Gaussian):   R = ", round(R_kernel, digits=3), "  (λ=", λ_kernel, ")   (paper reports R ≈ 0.74)")

**Kernel `R≈0.735` is very close to the paper's own `0.74`**: a clean,
direct match. **Naive `R≈0.11`** is small but positive, the same ballpark
as the paper's own `0.19` "extremely poor" naive case. The qualitative
story the paper is built on (kernel clearly, substantially beating
naive) holds here. Worth seeing directly:

In [ ]:
fig_pred = Figure(size=(900, 350))
ax = Axis(fig_pred[1, 1], xlabel="t", ylabel="x_L(t+0.5)", title="Free-standing prediction: naive vs. kernel (test window)")
lines!(ax, 1:length(Yte_kernel), vec(Yte_kernel); label="truth", linewidth=2)
lines!(ax, 1:length(Yhat_naive), vec(Yhat_naive); label="naive prediction", linestyle=:dash)
lines!(ax, 1:length(Yhat_kernel), vec(Yhat_kernel); label="kernel prediction", linestyle=:dash)
axislegend(ax; position=:rb)
save("FIGURES/swarmRC/lymburn_naive_vs_kernel_prediction.png", fig_pred)
fig_pred

### The parameter sweep, and how point B was validated

The paper's Figs 6-9 sweep `(Kr, Ka)` across five orders of magnitude and
identify a "critical" region (neither too rigid nor too disordered)
where the swarm responds best to the predator; `:critical` (used
throughout this section) is their own labelled "point B" from that
sweep, their best-performing point.

A 5-seed ensemble reproduction of the paper's Fig 8(c) (performance vs.
polarisation across the entire sweep grid, `N=200`, `(Kr,Ka)` on a `10×10`
log grid over `[10^{-3},10^2]`) gives **point B `R = 0.705 ± 0.027`**, and
the **peak `R` across the whole grid is `0.706`, landing at point B
itself**, both close to the paper's own reported `~0.74-0.8` there, and
consistent with the paper's own claim that point B is the sweep's
best-performing point. The low-`Φ_P` side of the arc (the B/C region)
reproduces the paper's shape cleanly. One open gap: the paper's Fig 8(c)
also shows a second, smaller rise in performance at *high* `Φ_P` (their
point A, high-`Kr` region) that this reproduction doesn't show; ours
stays flat/near zero out there.

**Sweep length matters more than sweep resolution here**: `train_len=3000,
predict_len=800` (used throughout this section) is well past the point of
diminishing returns for the *mean*. `train_len=4500, predict_len=1200`
only moves point B's mean `R` from `0.674` to `0.695` in a smaller
isolated check, while continuing to tighten the seed-to-seed spread
(`std` `0.018→0.007`). Shorter windows (`train_len=2000, predict_len=400`)
give a visibly noisier, lower estimate (point B `R≈0.53±0.04`, peak `R`
across the grid only `~0.6`); if a sweep result elsewhere looks
surprisingly low or noisy, check `train_len`/`predict_len` before
suspecting the model itself.

**`Φ_P` must be measured from the *driven* run, not an undriven pre-run**:
a swarm that looks fully rigid (`Φ_P≈1`) when undriven can still respond
normally once the predator is active, because the predator's forcing is
often strong enough to break a lock that only exists in the swarm's own
undriven dynamics.

That full validation is expensive (~3.6 hours for the 5-seed, `10×10`
grid at `train_len=3000, predict_len=800`) and isn't re-run live in this
notebook. The small demo below just shows the mechanism at a much
smaller scale, and the "Reproducing the paper's figures in full" section
further down loads the already-computed result. **To reproduce the full
validation yourself**: `(Kr,Ka)` grid
`10 .^ range(-3,2,length=10)` on each axis, `N=200`, paper-style
predator (as built above), `shift=200, train_len=3000, predict_len=800,
washout=200`, `M=200`, `Φ_P` measured from the driven run
(`measure_phiP_driven=true`), `nensemble=5`.

**Reference: the paper's actual parameters** (Sec III), for a full-scale
reproduction (also documented at the top of
`ABM/models/Lymburn/my_Lymburn_experiments.jl`):

| Parameter | Value |
|---|---|
| `N` | 200 |
| `Kr`, `Ka` (swept) | `[0.001, 100]`, pseudo-logarithmic (exact grid not specified) |
| `Kp` | 0 ("safe world") and 100 ("risky world", Lorenz-driven); run separately |
| `Kh`, `Kf` | 2, 20 (fixed) |
| `rr`, `ra`, `rp` | 1, 1, 2 (fixed) |
| `s`, `alpha`, `beta` | 10, 200, 0.1 (fixed) |
| `dt` | 0.02 |
| sim length | 5x10⁴ steps/grid point, discard first 1000 as transient |
| RC task | predict Lorenz-x 0.5 time units ahead; kernel layer `M=200` |
| ensembles | not stated exactly, but the paper's figures are themselves smoothed/averaged: `nensemble` in the 5-10+ range per grid point is a reasonable target, not 1 |

**Grid adequacy:** the live `6×6` grid below is only a plumbing/smoke test.
The saved `10×10`, five-seed Fig. 8(c) result has approximately the visible
point count needed to recover the paper's qualitative performance arc, but
it is not a grid-convergence study and is too coarse for precise phase
boundaries or peak locations across five decades. For a defensible smooth
parameter map, use at least `21×21` log-spaced points (roughly four points
per decade), retain 5-10+ seeds, and compare against a coarser grid before
claiming resolution independence.

All of these map directly onto `LymburnParams`/`Lymburn_params_from_preset`
fields or `run_Lymburn_rc_sweep` keyword arguments (including `nensemble`):
a paper-scale run is a matter of setting them to the values above and
being prepared to wait.

In [ ]:
include("ABM/my_ABM_experiments.jl")
include("ABM/models/Lymburn/my_Lymburn_experiments.jl")

# This 36-point, two-ensemble demonstration took more than one hour on the
# release-test machine. Type the phrase below to run it deliberately.
lymburn_sweep_confirmation = ""  # set to "RUN LYMBURN DEMO SWEEP"
run_lymburn_demo_sweep = lymburn_sweep_confirmation == "RUN LYMBURN DEMO SWEEP"

if run_lymburn_demo_sweep
    base_simcfg_sweep = SimulationConfig(steps=800, dt=dt, seed=1)
    base_params_sweep = Lymburn_params_from_preset(:driven; N=30)
    grid = (Kr=10.0 .^ range(-2, 2, length=6), Ka=10.0 .^ range(-2, 2, length=6))
    df_sweep = run_Lymburn_rc_sweep(
        base_simcfg_sweep, base_params_sweep, U[:, 1:1200], target[1:1200];
        grid=grid, shift=100, train_len=800, predict_len=200, washout=100,
        M=30, kneigh=5, n_repeats_consistency=3, transient_steps=200,
        nensemble=2, measure_phiP_driven=true, show_progress=true,
    )
    df_sweep_summary = summarise_experiment(df_sweep, [:Kr, :Ka];
        statcols=[:polarisation_mean, :polarisation_mean_driven,
                  :rotation_mean, :R, :Theta])
else
    println("Skipped the >1 hour Lymburn demonstration sweep. Set lymburn_sweep_confirmation=\"RUN LYMBURN DEMO SWEEP\" to reproduce it.")
end

In [ ]:
if run_lymburn_demo_sweep
    fig_sweep = plot_Lymburn_sweep_grid(df_sweep_summary;
        values=[:polarisation_mean_ens_mean, :rotation_mean_ens_mean,
                :R_ens_mean, :Theta_ens_mean])
    save("FIGURES/swarmRC/lymburn_sweep_heatmap.png", fig_sweep)
    fig_sweep
else
    display("text/html", "<img src='../FIGURES/swarmRC/lymburn_sweep_heatmap.png' alt='Saved Lymburn demonstration sweep' style='max-width:100%;'>")
end

Polarisation rising with `Ka` is exactly the trend the paper describes:
that part replicates cleanly even at this small, `N=30` demo scale.
This sweep is a mechanism demo (`nensemble=2`, short runs), not a
replication; see the full-validation recipe above for what that
actually takes.

### Reproducing the paper's figures in full

This is the actual validated reproduction of Fig. 8(c), not a cheap
stand-in. Two things a smaller/faster demo gets wrong, confirmed by
direct testing (not assumed): cutting swarm size down (even to `N=200`
with short runs) kills the arc, and cutting run length down (even at
`N=30` with the paper's full run length) *also* kills the arc. Neither
alone is the fix. **Both a large `N` and long runs are required
together**, because this swarm's own dynamics are intermittent
(documented throughout this notebook: order parameters that never
settle to a fixed value) and averaging that out needs real time, not
just a big swarm.

The actual computation (5 seeds, `10×10` grid, `N=200`,
`train_len=3000, predict_len=800`) takes ~3.6 hours, so it isn't re-run
live here. It is sufficient for the qualitative Fig. 8(c) arc, not for
resolution-converged phase boundaries. The cell below just loads the already-computed figure
(`FIGURES/swarmRC/lymburn_fig8c_full.png`), shown next to Lymburn et al.'s
own Fig. 8(c,d) for direct comparison. The exact code that produced it
is further down, documented but not executed, same convention used for
the animation cells elsewhere in this notebook.

**Lymburn et al. (2021), Fig. 8(c,d)** (original, performance vs. polarisation order; colour keys `(Kr,Ka)`):

See Fig. 8(c,d) in [Lymburn et al. (2021)](https://doi.org/10.1063/5.0039745). The source figure is not redistributed with this repository.

**This notebook's reproduction** of panel (c), point B and the grid-wide peak both landing at `R≈0.705-0.706` (see below):

![Fig 8c reproduction](../FIGURES/swarmRC/lymburn_fig8c_full.png)

This produced point B `R = 0.705 ± 0.027` and a grid-wide peak `R` of
`0.706` at point B itself (see the discussion above for how the
`train_len`/`predict_len` choice was validated, and the honest gap that
remains at high `Φ_P`). Raw sweep tables are not distributed. The code
below reproduces them in memory; allow about 3.6 hours for the full run.

The exact code that generated the saved figure (not executed here:
~3.6 hours):

```julia
grid_full = (Kr = 10.0 .^ range(-3, 2, length=10), Ka = 10.0 .^ range(-3, 2, length=10))
base_params_full = Lymburn_params_from_preset(:critical; N=200)   # Kr/Ka overridden per grid point below
shift_full, train_len_full, predict_len_full, washout_full = 200, 3000, 800, 200

df_full = run_Lymburn_rc_sweep(
    SimulationConfig(steps=4000, dt=dt, seed=1), base_params_full, U, target;
    grid=grid_full, shift=shift_full, train_len=train_len_full, predict_len=predict_len_full,
    washout=washout_full, M=200, kneigh=5, n_repeats_consistency=3, transient_steps=1500,
    nensemble=5, measure_phiP_driven=true, show_progress=true,
)

df_full_summary = summarise_experiment(df_full, [:Kr, :Ka];
    statcols=[:polarisation_mean_driven, :R, :Theta])

fig_fig8c_full = plot_Lymburn_performance_vs_polarisation(df_full_summary;
    phiP_col=:polarisation_mean_driven_ens_mean, Rcol=:R_ens_mean)
save("FIGURES/swarmRC/lymburn_fig8c_full.png", fig_fig8c_full)
fig_fig8c_full
```

Colour is a genuine 2D `(Kr, Ka)` key (see `_bivariate_KrKa_color` in
`ABM/models/Lymburn/my_Lymburn_experiments.jl`) rather than a single
colourbar, matching the paper's own Fig 8(c) inset convention of
varying colour independently with both swept parameters. It's an
approximation of that inset's look (its exact colormap isn't documented
in the paper), not a pixel-exact copy. The orientation now matches the
published key: `K_a` controls the bottom-to-top blue-to-red hue and `K_r`
controls the left-to-right lightening.

> Remaining paper-reproduction discrepancies and implementation variants
> are tracked in `TECHNICAL_NOTES.md`; they are deliberately kept
> out of the student workflow.

## D. Topology as a third lens: persistent homology and CROCKER plots

A third, independent notion of "topology" for a swarm, alongside the
order parameters and the interaction-network topology from Part B: run
persistent homology (Vietoris-Rips filtration) directly on the agents'
raw positions, with no reference to the interaction rules at all
(`TDA/my_TDA.jl`). This section applies it to the Couzin swarms built in
Part B.

This follows Topaz, Ziegelmeier & Halverson (2015), *Topological Data
Analysis of Biological Aggregation Models*, PLoS ONE 10(5), e0126383.
[doi:10.1371/journal.pone.0126383](https://doi.org/10.1371/journal.pone.0126383).
They introduce the **CROCKER plot** (Contour Realization Of Computed
k-dimensional hole Evolution in the Rips complex): the Betti number
`β_d(ε, t)` as a function of simulation time `t` and connection radius
`ε`, shown as a contour plot (values `≥5` lumped together as noise).
Their headline result, on a Vicsek flocking model and a D'Orsogna
attraction-repulsion swarm: CROCKER plots reveal dynamical events
(cluster coagulation and fragmentation, mill formation, loops appearing
and disappearing) that classical order parameters like polarisation miss
entirely, sometimes even when two runs' order-parameter traces look
nearly identical.

D.1-D.2 build up the machinery (persistence diagrams, Betti curves, a
first CROCKER plot); D.3 reproduces that headline result directly, on the
Couzin model already built in Part B.

#### D.1 Persistence diagrams: a single snapshot

Standalone build (this section doesn't depend on B1/B2 having run
first): a small periodic 2D Couzin swarm, the same `:milling` preset used
as B1's own demo run.

Persistent homology here is the expensive part of the pipeline (cost
grows roughly like the cube of the agent count per frame), so agent
counts, frame counts and the ε-grid are kept deliberately small
throughout this section.

**Caveat**: `out_d` is periodic, and plain Euclidean Vietoris-Rips
persistence is not boundary-correct on a periodic domain: two agents
close across the wrap are treated as far apart. Everything below
demonstrates the TDA machinery on that basis, not a boundary-correct
result; a toroidal-metric or unbounded-domain construction would be
needed for that.

In [ ]:
include("ABM/load_ABM.jl")
include("ABM/models/Couzin/load_Couzin.jl")
include("TDA/my_TDA.jl")

scenario_d = Couzin_params_from_preset(:milling; N=50, L=50.0, dt=0.1)
simcfg_d = SimulationConfig(steps=1000, dt=scenario_d.dt, seed=1)
out_d = simulate_Couzin_2d(simcfg_d, scenario_d.P)
(length(out_d.pos_hist), length(out_d.t))   # (frames stored, timepoints)

In [ ]:
ph = ph_snapshot(out_d; t=200, maxdim=2)   # one persistence diagram per homology dimension 0,1,2

dims_bd = [bd_lifetime(ph[dim + 1]) for dim in 0:2]   # (births, deaths, lifetimes) per dimension
all_deaths = reduce(vcat, [bd[2] for bd in dims_bd if !isempty(bd[2])]; init=Float64[])
maxval = isempty(all_deaths) ? 1.0 : maximum(all_deaths) * 1.1

fig_pd = Figure(size=(500, 500))
ax = Axis(fig_pd[1, 1], xlabel="birth", ylabel="death", title="Persistence diagram (t=200)")
plts, labels = [], String[]
for (dim, (B, Dd, L)) in enumerate(dims_bd)
    push!(plts, scatter!(ax, B, Dd)); push!(labels, "H$(dim - 1)")
end
push!(plts, lines!(ax, [0, maxval], [0, maxval]; color=(:gray, 0.7), linestyle=:dash, linewidth=2))
push!(labels, "birth = death")
axislegend(ax, plts, labels; position=:rb)
fig_pd

H0 = connected components (birth=0 always, death = scale at which two clusters merge); H1 = loops. Points further from the diagonal are longer-lived, more topologically significant features.

#### D.2 Betti curves and a first CROCKER plot

Scalar per-frame summaries first, then the *full* β(ε) curve at one
snapshot (how many independent components/loops exist as the connection
radius ε grows), then Betti curves stacked across *all* sampled frames:
Topaz et al.'s CROCKER plot, one heatmap per homology dimension, showing
how the swarm's topology evolves over `(ε, t)`.

In [ ]:
t_idxs_d = unique(round.(Int, range(1, length(out_d.pos_hist); length=25)))   # 25 sampled frames
feats_d = tda_summaries(out_d; t_idxs=t_idxs_d, maxdim=1)   # scalar summaries only (memory-light)

plot_tda_summaries(feats_d; series=[:tp1, :n1, :maxL1])

In [ ]:
εs_d = collect(range(0.0, 8.0; length=60))
bc_d = betti_curves_snapshot(out_d; t=200, εs=εs_d, maxdim=1)
plot_betti_curves_snapshot(bc_d; dims=[0, 1], layout=:rows)

In [ ]:
crock_d = crocker(out_d; t_idxs=t_idxs_d, εs=εs_d, maxdim=1, τ=0.5)
plot_crocker_pair_topaz(
    crock_d; dims=(0, 1),
    filled=(false, true), show_contours=(true, true),
    max_display_levels=(10, 5),
)

#### D.3 Replicating Topaz et al.'s headline result: CROCKER reveals what order parameters miss

Topaz et al.'s headline claim is that the CROCKER plot reveals dynamical
events (cluster coagulation and fragmentation, cycles appearing and
disappearing) that a scalar order-parameter trace doesn't capture, even
when that trace already looks settled. Check this directly: start a
`:milling` Couzin swarm from a random initial condition (not the
already-settled runs from B1/B2) and track both `Φ_R(t)` (Couzin's own
rotation order parameter, recorded automatically) and the CROCKER
matrices `β0(ε,t)`, `β1(ε,t)`, over the same organisation transient.

In [ ]:
scenario_topo = Couzin_params_from_preset(:milling; N=80, L=60.0, dt=0.1)
simcfg_topo = SimulationConfig(steps=1500, dt=scenario_topo.dt, seed=3)
out_topo = simulate_Couzin_2d(simcfg_topo, scenario_topo.P)

plot_order_parameters(out_topo)

`Φ_R` (rotation) crosses 0.5 and stays there from `t≈23` onward: by that classical standard, the mill looks formed only a sixth of the way into the run.

In [ ]:
εs_topo = collect(range(0.0, 12.0; length=40))
t_idxs_topo = unique(round.(Int, range(1, length(out_topo.pos_hist); length=30)))
crock_topo = crocker(out_topo; t_idxs=t_idxs_topo, εs=εs_topo, maxdim=1, τ=0.5)

plot_crocker_pair_topaz(
    crock_topo; dims=(0, 1),
    filled=(true, true), show_contours=(true, true),
    max_display_levels=(10, 5),
)

At `ε≈4.3` (comfortably inside the ε-grid above), `β0` keeps switching
between 1 and 5 separate connected components all the way to `t≈145`,
only settling to a single component in the very last sampled frame
(`t=150`), checked directly against the raw `β0(ε,t)` values, not read
off the plot by eye. The swarm's fine-scale spatial connectivity is
still splitting and re-merging for almost the entire run, well after
`Φ_R` alone would suggest it has settled: exactly the kind of structure
Topaz et al. report CROCKER plots reveal and order parameters miss.
`β1` stays consistently non-zero (typically 1-4) once the mill has
enough structure to show a hole at all, consistent with Couzin's own
description of the torus regime as a ring around an empty core; unlike
`β0`, its value doesn't track `Φ_R` closely either.

**Not covered here**: "vineyards" (tracking individual topological features
continuously through time, rather than frame-by-frame) and
Wasserstein/persistence-landscape-based comparison of regimes. Per
`DP/README.md`, vineyard tracking is an unimplemented design sketch
(`TDA/Vineyards.ipynb`), a future student project (T3-H2/T3-P5), not
working code today. `diagram_distance`/`distance_matrix_from_diagrams` (in
`TDA/my_TDA.jl`) *do* work today for comparing whole diagrams pairwise
(e.g. across different regimes or seeds) if you need that instead.

## Where to go from here

That's the tour: an ESN built and diagnosed from scratch, a swarm built
and simulated in 2D and 3D, the swarm plugged directly into the same
reservoir-training pipeline as the ESN, and a third, topological lens
(persistent homology, CROCKER plots) applied back onto that same swarm,
with each part verified against its own paper's published results along
the way.

Most cells here run in seconds. Multi-hour Couzin branches require exact
confirmation phrases, while the full Lymburn reproduction is documented
as non-executable code beside its saved artifacts, so Run All stays safe.
For real project work, deliberately unlock and scale those runs, and use
[`DP/README.md`](../README.md) as the reference: it has the full directory
map (core vs. optional files per pillar) and points to the specific
project codes (e.g. T1-S2, T1-H1, T3-H1, T3-P1) that extend exactly the
pieces demonstrated above.